<a href="https://colab.research.google.com/github/Tamanna0612/-Flyrank-internship-ML-Tamanna-/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamanna0612/-Flyrank-internship-ML-Tamanna-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: "Pages ranking on Page 1 with a CTR below 2% experience severe traffic decay within the next 60 days."

Methodology Question: How was "severe traffic decay" defined as a label? Was the 60-day outcome window strictly forward-looking, or did it overlap with the CTR measurement period? If they overlap, the claim suffers from data leakage.

Finding 2: "Dynamic momentum features (like position volatility) improve decline prediction accuracy by 15% over static features."

Methodology Question: Did the validation design use a strict client-holdout split (GroupShuffleSplit)? If a standard random split was used, the model might just be memorizing specific clients' volatility patterns rather than learning a universal SEO rule.

In [ ]:
import os, getpass, duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import precision_score

# Secure Token Setup
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your HF Token: ')
os.environ['HF_TOKEN'] = HF_TOKEN

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Safe Loading: Fetching Jan, Feb, Mar to ensure a full 90-day window exists
paths = [
    f"'{REL}/fact_content_daily_performance/month=2026-01/*.parquet'",
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet'",
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
]
TABLES = {'fact_daily': f"read_parquet([{', '.join(paths)}])"}

print("Setup complete. Data connection established.")


Setup complete. Data connection established.


## 2. My model under an honest split (before/after)
A random split (Before) mixes pages from the same client into both the training and testing sets. This allows the model to "cheat" by memorizing a specific client's seasonal traffic drops. An honest split (After) using GroupShuffleSplit groups by client_hash_id, ensuring the test set contains entirely unseen clients. The drop in performance shows the true, generalizable power of the model.

In [ ]:
# 1. Load Data
query = f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               COALESCE(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END), 0) AS imp_prev45,
               COALESCE(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END), 0)      AS clk_prev45,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END)       AS pos_prev45,
               SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 100
    )
    SELECT *, (clk_prev45 / imp_prev45) AS ctr_prev45 FROM windowed
"""
df = con.sql(query).df()
df['pos_prev45'] = df['pos_prev45'].fillna(100)
df['ctr_prev45'] = df['ctr_prev45'].fillna(0)
df['is_declining'] = (df['imp_last45'] < 0.8 * df['imp_prev45']).astype(int)

features = ['imp_prev45', 'clk_prev45', 'pos_prev45', 'ctr_prev45']
X = df[features]
y = df['is_declining']
groups = df['client_hash_id']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# 2. Before: Random Split (Dishonest)
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(X, y, test_size=0.25, random_state=42)
rf_rand = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_tr_rand, y_tr_rand)
score_rand = precision_at_k(rf_rand.predict_proba(X_te_rand)[:, 1], y_te_rand, 50)

# 3. After: Grouped Split (Honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_gss, X_te_gss = X.iloc[train_idx], X.iloc[test_idx]
y_tr_gss, y_te_gss = y.iloc[train_idx], y.iloc[test_idx]

rf_gss = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_tr_gss, y_tr_gss)
score_gss = precision_at_k(rf_gss.predict_proba(X_te_gss)[:, 1], y_te_gss, 50)

print("--- Precision@50 Comparison ---")
print(f"Before (Random Split) : {score_rand:.3f} (Inflated)")
print(f"After  (Grouped Split): {score_gss:.3f} (Honest Generalization)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Precision@50 Comparison ---
Before (Random Split) : 1.000 (Inflated)
After  (Grouped Split): 0.380 (Honest Generalization)


## 3. Leakage audit

I am re-running the leakage audit from Week 3 on the final feature set. I will intentionally include imp_last45 (impressions from the outcome window) into the training features. If the model is suddenly able to predict declines with near-perfect accuracy, it confirms the metric is a leak and must remain strictly excluded from the final pipeline.

In [ ]:
# Introduce Data Leakage (Adding imp_last45 to features)
leaky_features = features + ['imp_last45']
X_leaky = df[leaky_features]

X_tr_leak, X_te_leak = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

# Train Leaky Model
rf_leak = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_tr_leak, y_tr_gss)
leak_preds = rf_leak.predict(X_te_leak)
leak_precision = precision_score(y_te_gss, leak_preds, zero_division=0)

print("--- Leakage Audit ---")
print(f"Precision with LEAKY feature (imp_last45): {leak_precision:.3f}")
print("Conclusion: The leaky feature allows the model to cheat. It is strictly excluded from my final vector.")


--- Leakage Audit ---
Precision with LEAKY feature (imp_last45): 0.980
Conclusion: The leaky feature allows the model to cheat. It is strictly excluded from my final vector.


## 4. Claim rewrite

Original Bold Claim: "My model perfectly predicts which pages will lose traffic next month, guaranteeing a massive increase in content refresh ROI."

Rewritten Safe Claim: "Based on observed historical GSC data, the model provides directional decision-support for content teams. It identifies pages with a higher measured probability of traffic decline, serving as a prioritization tool rather than an absolute guarantee."

In [ ]:
print("Self-check complete:")
print("1. Findings critiqued constructively.")
print("2. Honest grouped split executed and compared.")
print("3. Leakage audit confirms safe boundaries.")
print("4. Claims rewritten using safe, measured language.")


Self-check complete:
1. Findings critiqued constructively.
2. Honest grouped split executed and compared.
3. Leakage audit confirms safe boundaries.
4. Claims rewritten using safe, measured language.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.